In [ ]:
if len(df_full) > N_TEST_SAMPLES:
    print("\n" + "="*70)
    print(f"PROIEZIONE SU DATASET COMPLETO ({len(df_full)} samples)")
    print("="*70)
    
    for model_name, metrics in results.items():
        inference_time_per_sample = metrics['inference_time_seconds'] / metrics['test_samples']
        emissions_per_sample = metrics['inference_emissions_kg'] / metrics['test_samples']
        
        projected_time = inference_time_per_sample * len(df_full)
        projected_emissions = emissions_per_sample * len(df_full)
        
        print(f"\n{model_name}:")
        print(f"  Tempo stimato inference: {projected_time:.2f} s ({projected_time/60:.2f} min)")
        print(f"  Emissioni stimate inference: {projected_emissions * 1000:.3f} g CO2")
        if metrics.get('training_emissions_kg'):
            total_projected = metrics['training_emissions_kg'] + projected_emissions
            print(f"  Totale (training + inference completa): {total_projected * 1000:.3f} g CO2")

print("\n" + "="*70)
print("ANALISI COMPLETATA!")
print("="*70)
print(f"Risultati salvati in: {OUTPUT_DIR}/")

## 9. Proiezione su Dataset Completo

In [ ]:
print("\n" + "="*70)
print("RIEPILOGO CARBON FOOTPRINT")
print("="*70)

for model_name, metrics in results.items():
    print(f"\n{model_name}:")
    print(f"  Training: {metrics.get('training_emissions_kg', 0) * 1000:.3f} g CO2")
    print(f"  Inference ({N_TEST_SAMPLES} samples): {metrics.get('inference_emissions_g', 0):.6f} g CO2")
    print(f"  Totale: {metrics.get('total_emissions_kg', 0) * 1000:.3f} g CO2")
    if metrics.get('f1_score'):
        print(f"  F1-Score: {metrics['f1_score']:.4f}")
    print(f"  Efficienza: {metrics['inference_time_ms_per_sample']:.3f} ms/sample")

## 8. Riepilogo Carbon Footprint

In [ ]:
print("\n" + "="*70)
print("TABELLA COMPARATIVA - EFFICIENCY & CARBON FOOTPRINT")
print("="*70)

comparison_df = pd.DataFrame([
    {
        "Model": name,
        "Features": metrics["num_features"],
        "Inference Time (ms/sample)": f"{metrics['inference_time_ms_per_sample']:.3f}",
        "Inference CO2 (g)": f"{metrics.get('inference_emissions_g', 0):.6f}",
        "Training Time (min)": f"{metrics.get('training_time_minutes', 'N/A'):.2f}" if metrics.get('training_time_minutes') else "N/A",
        "Training CO2 (g)": f"{metrics.get('training_emissions_kg', 0) * 1000:.3f}" if metrics.get('training_emissions_kg') else "N/A",
        "Total CO2 (g)": f"{metrics.get('total_emissions_kg', 0) * 1000:.3f}" if metrics.get('total_emissions_kg') else "N/A",
        "F1-Score": f"{metrics.get('f1_score', 0):.4f}" if metrics.get('f1_score') else "N/A",
        "ROC-AUC": f"{metrics.get('roc_auc', 0):.4f}" if metrics.get('roc_auc') else "N/A"
    }
    for name, metrics in results.items()
])

print(comparison_df.to_string(index=False))

# Salva tabella comparativa
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "comparison_table.csv"), index=False)
print(f"\n✓ Tabella comparativa salvata: {os.path.join(OUTPUT_DIR, 'comparison_table.csv')}")

## 7. Tabella Comparativa

In [ ]:
print("\n" + "="*70)
print("SALVATAGGIO RISULTATI")
print("="*70)

# Salva risultati individuali
for model_name, metrics in results.items():
    output_file = os.path.join(OUTPUT_DIR, f"{model_name.replace(' ', '_').lower()}_test_results.json")
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f"✓ {output_file}")

## 6. Salvataggio Risultati

In [ ]:
results = {}

for model_name, config in MODELS_CONFIG.items():
    print("\n" + "="*70)
    print(f"TEST MODELLO: {model_name}")
    print("="*70)
    
    # Carica modello
    model_file = config["model_file"]
    meta_file = config["meta_file"]
    
    if not os.path.exists(model_file):
        print(f"⚠ Modello {model_file} non trovato. Skip.")
        continue
    
    print(f"Caricamento modello da {model_file}...")
    model = CatBoostRegressor()
    model.load_model(model_file)
    
    # Carica metadata (se disponibile)
    training_meta = {}
    if os.path.exists(meta_file):
        with open(meta_file, 'r', encoding='utf-8') as f:
            training_meta = json.load(f)
        print(f"✓ Metadata caricato da {meta_file}")
    
    # Prepara dataset per questo modello
    df_model = df_test.copy()
    text_col = config["text_col"]
    
    # Determina la colonna di testo da usare nel modello
    if text_col:
        model_text_col = text_col
    else:
        model_text_col = None
    
    # Genera cheap features se necessario
    if config["needs_cheap"]:
        print(f"Generazione cheap features...")
        if model_name == "Heuristic-Only":
            cheap = df_model[text_col].apply(cheap_features_heuristic).apply(pd.Series)
        else:  # Hybrid Ensemble
            cheap = df_model[text_col].apply(cheap_features_hybrid).apply(pd.Series)
        df_model = pd.concat([df_model, cheap], axis=1)
        print(f"✓ Cheap features generate: {len(cheap.columns)} colonne")
    
    # Seleziona feature columns
    if config["needs_cheap"] and len(config["num_cols"]) == 0:
        # Solo cheap features (Heuristic-Only)
        if model_name == "Heuristic-Only":
            cheap_feature_names = [
                "char_length", "word_count", "avg_word_length", "num_uppercase",
                "num_digits", "num_spaces", "num_punctuation", "num_special_chars",
                "uppercase_ratio", "digit_ratio", "punctuation_ratio", "special_char_ratio",
                "num_newlines", "num_tabs", "unique_word_count", "unique_word_ratio",
                "num_td_keywords", "has_todo", "has_debt", "num_code_symbols",
                "code_symbol_ratio", "has_url", "has_path"
            ]
            num_cols = [c for c in cheap_feature_names if c in df_model.columns]
        else:
            num_cols = []
        
        num_cols = [c for c in num_cols if c not in [TARGET_COL, LABEL_COL]]
        
        # Gestisci NaN
        for c in num_cols:
            if c in df_model.columns:
                df_model[c] = df_model[c].replace([np.inf, -np.inf], np.nan)
                df_model[c] = df_model[c].fillna(0.0)
        
        feature_cols = num_cols
        text_features_list = []
        
    elif len(config["num_cols"]) == 0 and not config["needs_cheap"]:
        # Solo testo (Text-Based Baseline)
        feature_cols = [model_text_col]
        text_features_list = [model_text_col]
    else:
        # Testo + numeriche (Saliency-Enhanced o Hybrid)
        num_cols = [c for c in df_model.columns 
                   if any(c.startswith(p) for p in config["num_cols"])]
        
        num_cols = [c for c in num_cols if c not in [TARGET_COL, LABEL_COL]]
        
        # Gestisci NaN
        for c in num_cols:
            if c in df_model.columns:
                df_model[c] = df_model[c].replace([np.inf, -np.inf], np.nan)
                df_model[c] = df_model[c].fillna(0.0)
        
        feature_cols = [model_text_col] + num_cols
        text_features_list = [model_text_col]
    
    print(f"Feature utilizzate: {len(feature_cols)}")
    if text_features_list:
        print(f"  - Text features: {len(text_features_list)}")
        print(f"  - Numerical features: {len(feature_cols) - len(text_features_list)}")
    else:
        print(f"  - Numerical features only: {len(feature_cols)}")
    
    # Crea Pool
    test_pool = Pool(
        data=df_model[feature_cols],
        text_features=text_features_list if text_features_list else None
    )
    
    # Inference con carbon tracking
    print("\nInferenza con carbon tracking...")
    
    tracker = EmissionsTracker(
        project_name=f"test_{model_name.replace(' ', '_').lower()}",
        output_dir=OUTPUT_DIR,
        log_level="error"
    )
    tracker.start()
    
    start_time = time.time()
    pred_logit = model.predict(test_pool)
    inference_time = time.time() - start_time
    
    emissions = tracker.stop()
    
    # Converti logit a probabilità
    pred_prob = 1.0 / (1.0 + np.exp(-pred_logit))
    
    print(f"✓ Inferenza completata in {inference_time:.6f} secondi")
    print(f"  Tempo per campione: {(inference_time / len(df_model)) * 1000:.3f} ms")
    print(f"  Emissioni CO2 inference: {emissions:.9f} kg")
    
    # Calcolo metriche
    metrics = {
        "model_name": model_name,
        "num_features": len(feature_cols),
        "test_samples": len(df_model),
        "inference_time_seconds": inference_time,
        "inference_time_ms_per_sample": (inference_time / len(df_model)) * 1000,
        "inference_emissions_kg": float(emissions),
        "inference_emissions_g": float(emissions) * 1000,
    }
    
    # Training info (se disponibile dal metadata)
    if training_meta:
        metrics["training_time_minutes"] = training_meta.get("training_time_minutes", None)
        metrics["training_emissions_kg"] = training_meta.get("co2_emissions_kg", None)
        metrics["total_emissions_kg"] = (training_meta.get("co2_emissions_kg", 0) + emissions)
    
    # Metriche di distillation (se target disponibile)
    if has_target:
        y_true_logit = df_model[TARGET_COL].values
        rmse = np.sqrt(mean_squared_error(y_true_logit, pred_logit))
        mae = np.mean(np.abs(y_true_logit - pred_logit))
        
        metrics["rmse_logit"] = float(rmse)
        metrics["mae_logit"] = float(mae)
        
        print(f"  RMSE(logit): {rmse:.6f}")
        print(f"  MAE(logit): {mae:.6f}")
    
    # Metriche di classificazione (se label disponibile)
    if has_label:
        y_true = df_model[LABEL_COL].values
        
        # ROC-AUC e PR-AUC
        roc_auc = roc_auc_score(y_true, pred_prob)
        pr_auc = average_precision_score(y_true, pred_prob)
        
        # Usa soglia ottimale dal training metadata se disponibile
        threshold = training_meta.get("best_threshold", 0.5)
        y_pred = (pred_prob >= threshold).astype(int)
        
        # Metriche di classificazione
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        cm = confusion_matrix(y_true, y_pred)
        
        metrics["roc_auc"] = float(roc_auc)
        metrics["pr_auc"] = float(pr_auc)
        metrics["threshold"] = float(threshold)
        metrics["accuracy"] = float(acc)
        metrics["precision"] = float(prec)
        metrics["recall"] = float(rec)
        metrics["f1_score"] = float(f1)
        metrics["confusion_matrix"] = cm.tolist()
        metrics["true_negatives"] = int(cm[0, 0])
        metrics["false_positives"] = int(cm[0, 1])
        metrics["false_negatives"] = int(cm[1, 0])
        metrics["true_positives"] = int(cm[1, 1])
        
        print(f"  ROC-AUC: {roc_auc:.6f}")
        print(f"  PR-AUC: {pr_auc:.6f}")
        print(f"  F1-Score: {f1:.6f}")
        print(f"  Accuracy: {acc:.6f}")
        print(f"  Precision: {prec:.6f}")
        print(f"  Recall: {rec:.6f}")
        print(f"  Confusion Matrix:\n{cm}")
    
    results[model_name] = metrics

print("\n" + "="*70)
print("TEST COMPLETATI PER TUTTI I MODELLI")
print("="*70)

## 5. Test dei Modelli

In [ ]:
print("Caricamento dataset...")
df_full = pd.read_csv(TEST_FILE)
print(f"✓ Dataset caricato: {len(df_full)} righe totali")

# Pulizia del testo per tutte le colonne di testo
text_columns = df_full.select_dtypes(include=['object']).columns
for col in text_columns:
    if 'text' in col.lower() or col == 'orig_text':
        print(f"  Pulizia colonna: {col}")
        df_full[col] = df_full[col].apply(clean_text)

# Rimuovi righe con testo vuoto nella colonna principale
if 'orig_text' in df_full.columns:
    before = len(df_full)
    df_full = df_full[df_full['orig_text'].str.strip() != '']
    df_full = df_full[df_full['orig_text'].notna()]
    after = len(df_full)
    if before != after:
        print(f"  Rimossi {before - after} righe con testo vuoto")

# Seleziona le prime N_TEST_SAMPLES righe ripulite
df_test = df_full.head(N_TEST_SAMPLES).copy()
print(f"✓ Dataset di test preparato: {len(df_test)} righe")

# Verifica disponibilità colonne target
has_target = TARGET_COL in df_test.columns
has_label = LABEL_COL in df_test.columns

print(f"  Colonna {TARGET_COL}: {'✓' if has_target else '✗'}")
print(f"  Colonna {LABEL_COL}: {'✓' if has_label else '✗'}")
print(f"\nColonne disponibili nel dataset:")
print(df_test.columns.tolist())

## 4. Caricamento e Preparazione Dataset

In [ ]:
# Regex globali per cheap features
_re_word = re.compile(r"\b\w+\b", re.UNICODE)
_re_upper = re.compile(r"[A-Z]")
_re_digit = re.compile(r"\d")
_re_punct = re.compile(r"[.,;:!?]")
_re_special = re.compile(r"[^\w\s]")
_re_code_symbols = re.compile(r"[{}()\[\]<>=+\-*/]")

def cheap_features_heuristic(text: str) -> dict:
    """Genera cheap features per il modello Heuristic-Only."""
    t = "" if text is None else str(text)
    tl = t.lower()
    
    # Conta caratteri e parole
    char_length = len(t)
    words = _re_word.findall(t)
    word_count = len(words)
    unique_words = set(w.lower() for w in words)
    unique_word_count = len(unique_words)
    unique_word_ratio = unique_word_count / word_count if word_count > 0 else 0.0
    avg_word_length = sum(len(w) for w in words) / word_count if word_count > 0 else 0.0
    
    # Conta vari tipi di caratteri
    num_uppercase = len(_re_upper.findall(t))
    num_digits = len(_re_digit.findall(t))
    num_spaces = t.count(' ')
    num_punctuation = len(_re_punct.findall(t))
    num_special_chars = len(_re_special.findall(t))
    num_newlines = t.count('\n')
    num_tabs = t.count('\t')
    num_code_symbols = len(_re_code_symbols.findall(t))
    
    # Ratio
    uppercase_ratio = num_uppercase / char_length if char_length > 0 else 0.0
    digit_ratio = num_digits / char_length if char_length > 0 else 0.0
    punctuation_ratio = num_punctuation / char_length if char_length > 0 else 0.0
    special_char_ratio = num_special_chars / char_length if char_length > 0 else 0.0
    code_symbol_ratio = num_code_symbols / char_length if char_length > 0 else 0.0
    
    # Keywords TD
    has_todo = int('todo' in tl)
    has_debt = int('debt' in tl)
    
    # Conta keyword TD totali
    td_keywords = ['todo', 'fixme', 'hack', 'workaround', 'debt', 'refactor', 'temp', 'temporary']
    num_td_keywords = sum(tl.count(kw) for kw in td_keywords)
    
    # URL e path
    has_url = int('http://' in tl or 'https://' in tl or 'www.' in tl)
    has_path = int(('/' in t and '.' in t) or ('\\' in t))
    
    return {
        "char_length": char_length,
        "word_count": word_count,
        "avg_word_length": avg_word_length,
        "num_uppercase": num_uppercase,
        "num_digits": num_digits,
        "num_spaces": num_spaces,
        "num_punctuation": num_punctuation,
        "num_special_chars": num_special_chars,
        "uppercase_ratio": uppercase_ratio,
        "digit_ratio": digit_ratio,
        "punctuation_ratio": punctuation_ratio,
        "special_char_ratio": special_char_ratio,
        "num_newlines": num_newlines,
        "num_tabs": num_tabs,
        "unique_word_count": unique_word_count,
        "unique_word_ratio": unique_word_ratio,
        "num_td_keywords": num_td_keywords,
        "has_todo": has_todo,
        "has_debt": has_debt,
        "num_code_symbols": num_code_symbols,
        "code_symbol_ratio": code_symbol_ratio,
        "has_url": has_url,
        "has_path": has_path,
    }

# Keywords per Hybrid
KW = ["todo", "fixme", "hack", "workaround", "temp", "quick", "debt", "refactor"]

def kw_count_word(tl: str, kw: str) -> int:
    return len(re.findall(rf"\b{re.escape(kw)}\b", tl))

def cheap_features_hybrid(text: str) -> dict:
    """Genera cheap features per il modello Hybrid (con prefissi kw_, has_, etc.)."""
    t = "" if text is None else str(text)
    tl = t.lower()
    words = _re_word.findall(t)
    n_words = len(words)
    n_chars = len(t)
    n_upper = len(_re_upper.findall(t))
    n_digit = len(_re_digit.findall(t))
    n_punct = len(_re_punct.findall(t))
    
    feats = {
        "n_chars": n_chars,
        "n_words": n_words,
        "pct_upper": n_upper / n_chars if n_chars else 0.0,
        "pct_digit": n_digit / n_chars if n_chars else 0.0,
        "pct_punct": n_punct / n_chars if n_chars else 0.0,
        "has_should": int(" should " in f" {tl} "),
        "has_later": int(" later " in f" {tl} "),
        "has_for_now": int(" for now " in tl),
        "has_snake_case": 0,
        "has_camel_case": 0,
        "has_path": int(("/" in t) or ("\\" in t)),
        "has_equals": int("=" in t),
        "has_brackets": int(any(ch in t for ch in "(){}[]")),
    }
    
    for k in KW:
        cnt = kw_count_word(tl, k)
        feats[f"kw_{k}_cnt"] = cnt
    
    return feats

print("✓ Funzioni cheap_features_heuristic e cheap_features_hybrid definite")

## 3. Funzioni per Cheap Features

In [ ]:
def clean_text(text):
    """Pulisce il testo da spazi multipli e simboli inutili."""
    if pd.isna(text) or text is None:
        return ""
    
    text = str(text)
    # Rimuovi spazi multipli
    text = re.sub(r'\s+', ' ', text)
    # Rimuovi simboli di controllo e caratteri speciali inutili
    text = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    # Rimuovi simboli ripetuti eccessivamente (es. !!!!!!)
    text = re.sub(r'([!?.,;:]){3,}', r'\1\1', text)
    # Normalizza spazi attorno a punteggiatura
    text = re.sub(r'\s*([.,;:!?])\s*', r'\1 ', text)
    # Strip spazi iniziali/finali
    text = text.strip()
    
    return text

print("✓ Funzione clean_text definita")

## 2. Funzioni di Pulizia del Testo

In [ ]:
import os
import json
import time
import re
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import (
    mean_squared_error, roc_auc_score, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from codecarbon import EmissionsTracker

# ====== CONFIGURAZIONE ======
TEST_FILE = "2024_10IQR_technical debt.csv"
N_TEST_SAMPLES = 100
RANDOM_STATE = 42
OUTPUT_DIR = "test_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_COL = "logit_td"
LABEL_COL = "label"

# Modelli da testare
MODELS_CONFIG = {
    "Text-Based Baseline": {
        "model_file": "catboost_td_distilled.cbm",
        "meta_file": "catboost_td_distilled_meta.json",
        "text_col": "orig_text",
        "num_cols": [],
        "needs_cheap": False
    },
    "Saliency-Enhanced": {
        "model_file": "catboost_text_tokenaggr.cbm",
        "meta_file": "catboost_text_tokenaggr_meta.json",
        "text_col": "orig_text",
        "num_cols": ["sal_", "n_tokens_"],
        "needs_cheap": False
    },
    "Heuristic-Only": {
        "model_file": "catboost_td_cheap.cbm",
        "meta_file": "catboost_td_cheap_meta.json",
        "text_col": "orig_text",
        "num_cols": [],
        "needs_cheap": True
    },
    "Hybrid Ensemble": {
        "model_file": "catboost_td_full.cbm",
        "meta_file": "catboost_td_full_meta.json",
        "text_col": "orig_text",
        "num_cols": ["n_", "pct_", "kw_", "has_"],
        "needs_cheap": True
    }
}

print(f"✓ Configurazione caricata")
print(f"  Dataset: {TEST_FILE}")
print(f"  Numero campioni test: {N_TEST_SAMPLES}")
print(f"  Modelli configurati: {len(MODELS_CONFIG)}")

## 1. Import e Configurazione

# Test dei 4 Modelli CatBoost con Carbon Footprint

Questo notebook testa i seguenti modelli su un dataset di 100 campioni ripuliti:
1. **Text-Based Baseline** - Solo testo
2. **Saliency-Enhanced** - Testo + saliency features
3. **Heuristic-Only** - Solo cheap features euristiche
4. **Hybrid Ensemble** - Testo + cheap features

Metriche misurate:
- Tempo di inferenza
- Emissioni CO2
- Accuratezza, F1-Score, ROC-AUC
- Confronto prestazioni vs carbon footprint